# Week 13 — NLP Foundations
### Text Preprocessing, TF-IDF Vectorization & Spam Classification
**Next Gen Software Hub Pvt. Ltd. — AI & ML Internship**


In [ ]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df = pd.read_csv("datasets/sms_spam_text_data.csv")
print(f"Total texts: {df.shape[0]} | Spam ratio: {(df['Label'] == 'spam').mean()*100:.1f}%")
df.head()


## 1. Text Preprocessing Pipeline


In [ ]:
def clean_text(text):
    text = text.lower() # lowercase
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # remove URLs
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text) # remove punctuation
    text = re.sub(r'\d+', '', text) # remove numbers
    text = text.strip()
    return text

df["Clean_Text"] = df["Text"].apply(clean_text)
df[["Text", "Clean_Text"]].head()


## 2. TF-IDF Vectorization & Model Training


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["Clean_Text"], df["Label"], test_size=0.25, random_state=42, stratify=df["Label"]
)

vectorizer = TfidfVectorizer(max_features=500, stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF Vocabulary Size: {len(vectorizer.get_feature_names_out())}")

clf = MultinomialNB()
clf.fit(X_train_tfidf, y_train)
preds = clf.predict(X_test_tfidf)

print(classification_report(y_test, preds))


## 3. Real-Time Inference Demo


In [ ]:
def predict_sms(message):
    cleaned = clean_text(message)
    vec = vectorizer.transform([cleaned])
    pred = clf.predict(vec)[0]
    prob = clf.predict_proba(vec).max()
    return pred, prob

sample_msg = "Congratulations! Claim your free $500 gift card today."
result, confidence = predict_sms(sample_msg)
print(f"Message : '{sample_msg}'")
print(f"Result  : {result.upper()} (Confidence: {confidence*100:.1f}%)")
